# Phase 12 — La ville et l'heure

## Objectifs

- Utiliser la ville dans le modèle sans créer une colonne par ville.
- Encoder l'heure de manière cyclique afin que 23 h soit proche de 0 h.
- Normaliser la colonne `shape` et regrouper les formes rares.
- Mesurer la largeur du tableau de variables avant et après encodage.
- Vérifier les distances entre 23 h, 0 h et 20 h dans l'encodage cyclique.


## 1. Imports

In [ ]:
from pathlib import Path
import csv
import math
import re

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score
from sklearn.pipeline import Pipeline


## 2. Chemins et paramètres

In [ ]:
DATA_PATH = Path("../data/releves_klaxo3.csv")
OUTPUT_DIR = Path("../outputs")
PHASE12_DIR = OUTPUT_DIR / "phase_12_ville_heure_forme"
PHASE12_DIR.mkdir(parents=True, exist_ok=True)

COLUMNS = [
    "datetime", "city", "state", "country", "shape",
    "duration_seconds", "duration_hours_min", "comments",
    "date_posted", "latitude", "longitude",
]

MOTS_CLES_CANULAR = [
    "hoax", "fake", "prank", "joke",
    "not real", "made up", "fraud",
]

TEST_SIZE = 0.20
RANDOM_STATE = 42
SEUIL_VILLE_RARE = 5
SEUIL_FORME_RARE = 5


## 3. Chargement robuste

In [ ]:
lignes_valides = []
lignes_problemes = []

with open(DATA_PATH, "r", encoding="utf-8", errors="replace", newline="") as f:
    reader = csv.reader(f)
    for numero_ligne, row in enumerate(reader, start=1):
        if len(row) == len(COLUMNS):
            lignes_valides.append(row)
        else:
            lignes_problemes.append({
                "numero_ligne": numero_ligne,
                "nb_champs": len(row),
                "contenu": row,
            })

df = pd.DataFrame(lignes_valides, columns=COLUMNS)
print(f"Lignes chargées : {len(df)}")


## 4. Conversions et création de la cible

In [ ]:
for col in ["duration_seconds", "latitude", "longitude"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

for col in ["datetime", "date_posted"]:
    df[col] = pd.to_datetime(df[col], errors="coerce")

for col in ["city", "state", "country", "shape", "comments"]:
    df[col] = df[col].astype("string").str.strip().replace("", pd.NA)

pattern_canular = "|".join(re.escape(mot) for mot in MOTS_CLES_CANULAR)
df["comments_clean"] = df["comments"].fillna("").astype(str).str.lower()
df["is_hoax"] = df["comments_clean"].str.contains(
    pattern_canular, regex=True, na=False
).astype(int)


## 5. Découpage temporel

Le train est constitué des observations anciennes et le test des observations récentes. Les règles de normalisation dépendant des fréquences des villes ou des formes seront apprises uniquement sur le train.

In [ ]:
df_temporel = df.loc[df["datetime"].notna()].copy()
df_temporel = df_temporel.sort_values("datetime").copy()

position_coupure = int(len(df_temporel) * (1 - TEST_SIZE))
date_coupure = df_temporel.iloc[position_coupure]["datetime"]

df_train = df_temporel.loc[df_temporel["datetime"] < date_coupure].copy()
df_test = df_temporel.loc[df_temporel["datetime"] >= date_coupure].copy()

assert df_train["datetime"].max() < df_test["datetime"].min()
print(f"Date de coupure : {date_coupure}")


## 6. Analyse des villes

Le nombre de villes qui apparaissent une seule fois est calculé sur toute la transmission, comme demandé. La règle de regroupement des villes rares est ensuite apprise sur le train uniquement afin d'éviter toute fuite depuis le test.

In [ ]:
df["city_normalisee"] = (
    df["city"]
    .fillna("<MANQUANT>")
    .astype(str)
    .str.lower()
    .str.strip()
)

compte_villes_global = df["city_normalisee"].value_counts()
nombre_villes_total = int(compte_villes_global.size)
nombre_villes_une_fois = int((compte_villes_global == 1).sum())

print(f"Nombre de villes distinctes : {nombre_villes_total}")
print(f"Nombre de villes présentes une seule fois : {nombre_villes_une_fois}")


## 7. Apprentissage de la règle des villes rares sur le train

Les villes apparaissant moins de `SEUIL_VILLE_RARE` fois dans le train sont regroupées dans la catégorie `<VILLE_RARE>`. Les villes jamais vues lors de l'entraînement seront également traitées comme rares au moment du test.

In [ ]:
df_train["city_normalisee"] = (
    df_train["city"]
    .fillna("<MANQUANT>")
    .astype(str)
    .str.lower()
    .str.strip()
)

df_test["city_normalisee"] = (
    df_test["city"]
    .fillna("<MANQUANT>")
    .astype(str)
    .str.lower()
    .str.strip()
)

compte_villes_train = df_train["city_normalisee"].value_counts()
villes_conservees = set(
    compte_villes_train[compte_villes_train >= SEUIL_VILLE_RARE].index
)

def regrouper_ville(ville):
    if ville == "<MANQUANT>":
        return "<MANQUANT>"
    if ville in villes_conservees:
        return ville
    return "<VILLE_RARE>"

df_train["city_modele"] = df_train["city_normalisee"].apply(regrouper_ville)
df_test["city_modele"] = df_test["city_normalisee"].apply(regrouper_ville)

print(f"Villes conservées dans le train : {len(villes_conservees)}")
print(f"Catégories finales de ville dans train : {df_train['city_modele'].nunique()}")


## 8. Normalisation de la colonne `shape`

Les orthographes visiblement équivalentes sont regroupées avant le calcul des formes rares. Cette normalisation est déterministe et ne dépend pas de la cible.

In [ ]:
CORRECTIONS_FORMES = {
    "disk": "disc",
    "cigar": "cylinder",
    "changed": "changing",
}

def normaliser_forme(valeur):
    if pd.isna(valeur):
        return "<MANQUANT>"

    forme = str(valeur).lower().strip()
    if forme == "":
        return "<MANQUANT>"

    return CORRECTIONS_FORMES.get(forme, forme)

df_train["shape_normalisee"] = df_train["shape"].apply(normaliser_forme)
df_test["shape_normalisee"] = df_test["shape"].apply(normaliser_forme)

print("Formes après normalisation, avant regroupement des formes rares :")
print(df_train["shape_normalisee"].value_counts())


## 9. Apprentissage de la règle des formes rares sur le train

Les formes apparaissant moins de `SEUIL_FORME_RARE` fois dans le train sont regroupées dans `<FORME_RARE>`. Cette décision est déterminée à partir du train seul.

In [ ]:
compte_formes_train = df_train["shape_normalisee"].value_counts()
formes_conservees = set(
    compte_formes_train[compte_formes_train >= SEUIL_FORME_RARE].index
)

def regrouper_forme(forme):
    if forme == "<MANQUANT>":
        return "<MANQUANT>"
    if forme in formes_conservees:
        return forme
    return "<FORME_RARE>"

df_train["shape_modele"] = df_train["shape_normalisee"].apply(regrouper_forme)
df_test["shape_modele"] = df_test["shape_normalisee"].apply(regrouper_forme)

nombre_formes_final = int(df_train["shape_modele"].nunique())
print(f"Nombre de formes finales dans le train : {nombre_formes_final}")
print(df_train["shape_modele"].value_counts())


## 10. Encodage cyclique de l'heure

L'heure est encodée par deux coordonnées : sinus et cosinus. Cet encodage représente l'heure sur un cercle et évite de considérer que 23 h et 0 h sont éloignées.

In [ ]:
def ajouter_encodage_heure(dataframe):
    resultat = dataframe.copy()
    resultat["observation_year"] = resultat["datetime"].dt.year
    resultat["observation_month"] = resultat["datetime"].dt.month
    resultat["observation_hour"] = resultat["datetime"].dt.hour

    angle = 2 * np.pi * resultat["observation_hour"] / 24
    resultat["heure_sin"] = np.sin(angle)
    resultat["heure_cos"] = np.cos(angle)

    return resultat

df_train = ajouter_encodage_heure(df_train)
df_test = ajouter_encodage_heure(df_test)


## 11. Vérification des distances entre heures

La distance euclidienne est calculée dans l'espace `(sin, cos)`. La distance entre 23 h et 0 h doit être plus petite que la distance entre 23 h et 20 h.

In [ ]:
def vecteur_heure(heure):
    angle = 2 * math.pi * heure / 24
    return np.array([math.sin(angle), math.cos(angle)])

distance_23_0 = float(
    np.linalg.norm(vecteur_heure(23) - vecteur_heure(0))
)

distance_23_20 = float(
    np.linalg.norm(vecteur_heure(23) - vecteur_heure(20))
)

print(f"Distance entre 23 h et 0 h : {distance_23_0:.6f}")
print(f"Distance entre 23 h et 20 h : {distance_23_20:.6f}")

assert distance_23_0 < distance_23_20


## 12. Construction du texte compact

La ville et la forme traitées sont intégrées dans un texte qui sera vectorisé par TF-IDF. Cette approche évite de créer une colonne binaire par ville. Le vocabulaire est appris sur le train uniquement par le pipeline.

In [ ]:
def construire_texte_modele(dataframe):
    resultat = dataframe.copy()
    resultat["text_features_phase12"] = (
        "city " + resultat["city_modele"].astype(str)
        + " state " + resultat["state"].fillna("<MANQUANT>").astype(str)
        + " country " + resultat["country"].fillna("<MANQUANT>").astype(str)
        + " shape " + resultat["shape_modele"].astype(str)
    )
    return resultat

df_train = construire_texte_modele(df_train)
df_test = construire_texte_modele(df_test)


## 13. Largeur du tableau avant et après traitement

La largeur avant traitement correspond aux variables numériques utilisées avant vectorisation. La largeur après traitement est obtenue après ajustement du préprocesseur sur le train. Elle comprend le vocabulaire TF-IDF limité, les variables numériques et l'encodage cyclique de l'heure.

In [ ]:
features_numeriques = [
    "duration_seconds",
    "latitude",
    "longitude",
    "observation_year",
    "observation_month",
    "heure_sin",
    "heure_cos",
]

features_modele = ["text_features_phase12"] + features_numeriques

X_train = df_train[features_modele].copy()
X_test = df_test[features_modele].copy()
y_train = df_train["is_hoax"].copy()
y_test = df_test["is_hoax"].copy()

largeur_avant = len(features_modele)
print(f"Largeur avant vectorisation : {largeur_avant}")


## 14. Pipeline et modèle

Le vocabulaire de ville et de forme est appris dans TF-IDF sur le train. Les médianes d'imputation sont également calculées sur le train uniquement.

In [ ]:
preprocessing = ColumnTransformer(
    transformers=[
        (
            "texte",
            TfidfVectorizer(
                lowercase=True,
                min_df=2,
                max_features=500,
                ngram_range=(1, 1),
            ),
            "text_features_phase12",
        ),
        (
            "numerique",
            SimpleImputer(strategy="median"),
            features_numeriques,
        ),
    ]
)

modele_phase12 = Pipeline(
    steps=[
        ("preprocessing", preprocessing),
        ("classifier", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        )),
    ]
)

modele_phase12.fit(X_train, y_train)

matrice_train_transformee = modele_phase12.named_steps["preprocessing"].transform(X_train)
largeur_apres = int(matrice_train_transformee.shape[1])

print(f"Largeur après prétraitement : {largeur_apres}")


## 15. Évaluation du modèle

In [ ]:
y_pred = modele_phase12.predict(X_test)

precision_phase12 = precision_score(y_test, y_pred, zero_division=0)
recall_phase12 = recall_score(y_test, y_pred, zero_division=0)
accuracy_phase12 = accuracy_score(y_test, y_pred)

print(f"Precision : {precision_phase12:.2%}")
print(f"Recall : {recall_phase12:.2%}")
print(f"Accuracy : {accuracy_phase12:.2%}")


## 16. Matrice de confusion

In [ ]:
matrice_phase12 = confusion_matrix(y_test, y_pred)

df_matrice_phase12 = pd.DataFrame(
    matrice_phase12,
    index=["Réel : non-canular", "Réel : canular"],
    columns=["Prédit : non-canular", "Prédit : canular"],
)

df_matrice_phase12


## 17. Export des résultats

In [ ]:
resume_phase12 = pd.DataFrame([
    {
        "nombre_villes_distinctes": nombre_villes_total,
        "nombre_villes_une_fois": nombre_villes_une_fois,
        "seuil_ville_rare": SEUIL_VILLE_RARE,
        "villes_conservees_train": len(villes_conservees),
        "categories_ville_finales_train": df_train["city_modele"].nunique(),
        "formes_finales_train": nombre_formes_final,
        "seuil_forme_rare": SEUIL_FORME_RARE,
        "largeur_avant": largeur_avant,
        "largeur_apres": largeur_apres,
        "distance_23_0": distance_23_0,
        "distance_23_20": distance_23_20,
        "precision": precision_phase12,
        "recall": recall_phase12,
        "accuracy": accuracy_phase12,
        "date_coupure": date_coupure,
    }
])

resume_phase12.to_csv(
    PHASE12_DIR / "resume_phase12.csv",
    index=False,
)

df_matrice_phase12.to_csv(
    PHASE12_DIR / "matrice_confusion_phase12.csv",
    index=True,
)

compte_villes_global.rename_axis("ville").reset_index(name="nombre_releves").to_csv(
    PHASE12_DIR / "frequences_villes_globales.csv",
    index=False,
)

df_train[["shape", "shape_normalisee", "shape_modele"]].drop_duplicates().to_csv(
    PHASE12_DIR / "normalisation_formes.csv",
    index=False,
)

print(PHASE12_DIR)
